# Microsoft Agent Framework — Azure OpenAI (Atsakymų API)

Šiame kodo pavyzdyje naudosite **Microsoft Agent Framework (MAF)**, kad sukurtumėte paprastą agentą, pagrįstą **Azure OpenAI**, naudojant **Atsakymų API**.

> **Migracijos pastaba:** Šis pavyzdys anksčiau naudojo Semantic Kernel su GitHub modeliais. Jis buvo perkeliamas į Microsoft Agent Framework, o GitHub modeliai (pasenę, nutraukiami 2026 m. liepą) buvo pakeisti Azure OpenAI, kuris palaiko Atsakymų API. `OpenAIChatClient` MAF taikosi į Azure OpenAI stabilų `/openai/v1/` galinį tašką ir numatytasis naudoja Atsakymų API.

Šio pavyzdžio tikslas – parodyti veiksmus, kurie vėliau bus taikomi kituose kodo pavyzdžiuose įgyvendinant įvairius agentinius modelius.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Importuokite reikiamas Python bibliotekas


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Įrankio apibrėžimas

Microsoft Agent Framework sistemoje **įrankis** yra paprasta Python funkcija, dekoruota `@tool`, kurią agentas gali iškviesti. Žemiau apibrėžiame įrankį, kuris pateikia atsitiktinę atostogų vietą ir vengia kartoti ankstesnę.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Agentų kūrimas

Čia mes kuriame Agentą pavadinimu `TravelAgent`.

Šiame pavyzdyje naudojame labai paprastas instrukcijas. Drąsiai keiskite šias instrukcijas, kad pamatytumėte, kaip keičiasi agento elgsena.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Agento paleidimas

Dabar galime paleisti agentą. Sukuriame `AgentSession`, kad agentas prisimintų pokalbį per įvairius veiksmus, tada siunčiame du `user_inputs`. Pirmasis prašo kelionės; antrasis sako, kad naudotojui nepatiko pasiūlymas ir prašo kito — agentas naudoja sesijos istoriją bei įrankį `get_random_destination`, kad atsakytų.

Galite pakeisti šias žinutes, kad pamatytumėte, kaip agentas reaguoja skirtingai. Atsakymai **transliuojami** žodis po žodžio.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
